# Week 1 quickstart

Runs the whole week-1 pipeline on the **real** data: download → clean → features → benchmarks → Poisson → Setup A models.

**Colab:** put your repo URL in the first cell, then *Runtime → Run all*. **Local:** skip the first cell and start Jupyter from the repo root.

In [2]:
# Colab only: clone the repo and install packages
REPO_URL = "https://github.com/<your-username>/football-prediction.git"   # <-- change this
import os, sys
if "google.colab" in sys.modules and not os.path.exists("football-prediction"):
    !git clone $REPO_URL
    %cd football-prediction
    !pip install -q -r requirements.txt
elif os.path.basename(os.getcwd()) == "notebooks":
    %cd ..

c:\Users\PCN\Projects\FootballPrediction


## 1. Download and clean (Lyzing)
Read the sanity-check output. Every season should have 380 matches and 20 teams. Look through the team-name list for duplicates.

In [3]:
from src.load_data import main as load_main, read_matches
load_main()
m = read_matches()
m.head()

matches per season (expect 380): all OK
teams per season (expect 20): all OK
missing results: 0
missing shots-on-target: 0
missing odds: 0
all team names (check for duplicates/typos):
['Arsenal', 'Aston Villa', 'Birmingham', 'Blackburn', 'Blackpool', 'Bolton', 'Bournemouth', 'Brentford', 'Brighton', 'Burnley', 'Cardiff', 'Charlton', 'Chelsea', 'Crystal Palace', 'Derby', 'Everton', 'Fulham', 'Huddersfield', 'Hull', 'Ipswich', 'Leeds', 'Leicester', 'Liverpool', 'Luton', 'Man City', 'Man United', 'Middlesbrough', 'Newcastle', 'Norwich', "Nott'm Forest", 'Portsmouth', 'QPR', 'Reading', 'Sheffield United', 'Southampton', 'Stoke', 'Sunderland', 'Swansea', 'Tottenham', 'Watford', 'West Brom', 'West Ham', 'Wigan', 'Wolves']
saved 7600 matches -> data\processed\matches.csv


,match_id,season,date,home,away,hg,ag,result,label,hs,as_,hst,ast,odds_h,odds_d,odds_a
0,0,0506,2005-08-13,Aston Villa,Bolton,2,2,D,1,3.0,13.0,2.0,6.0,2.30,3.25,3.00
1,1,0506,2005-08-13,Everton,Man United,0,2,A,2,10.0,12.0,5.0,5.0,5.00,3.40,1.72
2,2,0506,2005-08-13,Fulham,Birmingham,0,0,D,1,15.0,7.0,7.0,4.0,2.37,3.25,2.87
3,3,0506,2005-08-13,Man City,West Brom,0,0,D,1,15.0,13.0,8.0,3.0,1.72,3.40,5.00
4,4,0506,2005-08-13,Middlesbrough,Liverpool,0,0,D,1,4.0,16.0,2.0,7.0,2.87,3.20,2.40


## 2. Features (Darareach, Lyzing)
Pick 3 random matches and check a few values by hand against earlier rows (leakage check).

In [4]:
from src.features import build_features, SETUPS
f = build_features(m)
f.to_csv("data/processed/features.csv", index=False)
print("missing in setup C:", int(f[SETUPS['C']].isna().sum().sum()))
f.sample(3, random_state=1)[["date", "home", "away", "result", "elo_diff", "home_form_pts5", "away_form_pts5", "home_rest_days", "h2h_score", "h2h_n"]]

missing in setup C: 0


,date,home,away,result,elo_diff,home_form_pts5,away_form_pts5,home_rest_days,h2h_score,h2h_n
7024,2023-12-27,Chelsea,Crystal Palace,H,26.734643,1.2,0.6,3.0,0.833333,6
4536,2017-05-12,West Brom,Chelsea,A,-208.911243,0.2,2.4,6.0,0.388889,6
1825,2010-03-24,Man City,Everton,A,-3.546913,1.8,2.0,3.0,0.388889,6


## 3. Benchmarks (Cheypol)
Copy these rows into the Results Log. Benchmarks can be scored on test any time, because nothing is tuned.

In [5]:
from src.evaluate import benchmark_table, split, evaluate
bench = benchmark_table(f)
bench

,model,split,accuracy,macro_F1,brier,rps
0,always home,val,0.4842,0.2175,1.0316,0.4013
1,frequencies,val,0.4842,0.2175,0.6328,0.2280
2,bookmaker (B365),val,0.5579,0.4058,0.5735,0.1990
3,always home,test,0.4342,0.2018,1.1316,0.4507
4,frequencies,test,0.4342,0.2018,0.6469,0.2348
5,bookmaker (B365),test,0.5684,0.4222,0.5559,0.1906


## 4. Walk-forward Poisson (Chhoungkorng)
Takes about a minute. These probabilities are the Poisson baseline and the extra Setup D features.

In [6]:
from src.poisson import walk_forward_probs, fill_missing_probs
probs = walk_forward_probs(m)
probs.to_csv("data/processed/poisson_probs.csv", index=False)
print("unseen-team matches filled with frequencies:", int(probs.pois_missing.sum()))
f = f.merge(fill_missing_probs(probs), on="match_id", how="left")
_, val, _ = split(f)
evaluate(val.label, val[["pois_pH", "pois_pD", "pois_pA"]].values)

unseen-team matches filled with frequencies: 126


{'accuracy': 0.5184, 'macro_F1': 0.379, 'brier': 0.6101, 'rps': 0.2164}

## 5. Setup A on validation (Cheypol)
Default settings only. Tuning comes in week 3. **Do not touch the test split here.**

In [7]:
from src.models import run_validation
run_validation(f, {"A": SETUPS["A"]})

,model,setup,split,accuracy,macro_F1,brier,rps
0,LR,A,val,0.5316,0.3823,0.5861,0.2047
1,RF,A,val,0.5184,0.4301,0.6013,0.2107
2,XGB,A,val,0.4816,0.4021,0.6660,0.2336


## Next (week 2)
- Look at the setup A results above next to the benchmarks. Above ~60% accuracy means check for leakage.
- `run_validation(f, SETUPS)` runs all four setups once features are checked.
- Run `python -m pytest -q` after any change to `src/features.py`.

In [8]:
for name, kw in [("plain", {}),
                 ("time decay", {"xi": 0.0019}),
                 ("Dixon-Coles", {"xi": 0.0019, "dixon_coles": True})]:
    p = walk_forward_probs(m, **kw).merge(m[["match_id", "season", "label"]], on="match_id")
    _, v, _ = split(p)
    print(name, evaluate(v.label, v[["pois_pH", "pois_pD", "pois_pA"]].fillna(1/3).values),
          "| avg rho:", round(p.pois_rho.mean(), 3))

plain {'accuracy': 0.5184, 'macro_F1': 0.379, 'brier': 0.612, 'rps': 0.2172} | avg rho: 0.0
time decay {'accuracy': 0.5263, 'macro_F1': 0.3835, 'brier': 0.6058, 'rps': 0.2141} | avg rho: 0.0
Dixon-Coles {'accuracy': 0.5263, 'macro_F1': 0.3835, 'brier': 0.606, 'rps': 0.2141} | avg rho: -0.044


In [9]:
results = run_validation(f, SETUPS)
results

,model,setup,split,accuracy,macro_F1,brier,rps
0,LR,A,val,0.5316,0.3823,0.5861,0.2047
1,RF,A,val,0.5184,0.4301,0.6013,0.2107
2,XGB,A,val,0.4816,0.4021,0.6660,0.2336
3,LR,B,val,0.5342,0.3927,0.5765,0.2005
4,RF,B,val,0.5447,0.4285,0.5811,0.2018
5,XGB,B,val,0.4974,0.4318,0.6492,0.2269
6,LR,C,val,0.5421,0.3992,0.5759,0.1998
7,RF,C,val,0.5395,0.4201,0.5783,0.2006
8,XGB,C,val,0.4658,0.3962,0.6582,0.2282
9,LR,D,val,0.5395,0.3997,0.5827,0.2029


In [10]:
probs_dc = walk_forward_probs(m, xi=0.0019, dixon_coles=True)
probs_dc.to_csv("data/processed/poisson_probs.csv", index=False)
f = f.drop(columns=[c for c in f.columns if c.startswith("pois_")])
f = f.merge(fill_missing_probs(probs_dc), on="match_id", how="left")

In [11]:
import numpy as np, pandas as pd
from src.models import fit_predict
seasons = ["1819", "1920", "2021", "2122", "2223"]
rows = []
for s, cols in SETUPS.items():
    for name in ["LR", "RF", "XGB"]:
        Ps, ys = [], []
        for vs in seasons:
            tr = f[(f.season >= "1011") & (f.season < vs)]
            te = f[f.season == vs]
            _, P = fit_predict(name, tr, te, cols)
            Ps.append(P); ys.append(te.label.values)
        rows.append({"model": name, "setup": s, **evaluate(np.concatenate(ys), np.vstack(Ps))})
roll = pd.DataFrame(rows)
print(roll.pivot(index="model", columns="setup", values="rps").round(4))
print(roll.pivot(index="model", columns="setup", values="accuracy").round(4))

setup       A       B       C       D
model                                
LR     0.2100  0.2030  0.2028  0.2027
RF     0.2208  0.2090  0.2078  0.2077
XGB    0.2430  0.2355  0.2342  0.2369
setup       A       B       C       D
model                                
LR     0.5268  0.5437  0.5432  0.5453
RF     0.4837  0.5258  0.5295  0.5289
XGB    0.4626  0.4863  0.4900  0.4942


In [12]:
from src.evaluate import bookmaker_probs, frequencies
v5 = f[f.season.isin(seasons)]
tr5 = f[(f.season >= "1011") & (f.season < "1819")]
print("bookmaker  ", evaluate(v5.label, bookmaker_probs(v5)))
print("poisson DC ", evaluate(v5.label, v5[["pois_pH", "pois_pD", "pois_pA"]].values))
print("frequencies", evaluate(v5.label, frequencies(tr5.label, len(v5))))

bookmaker   {'accuracy': 0.5547, 'macro_F1': 0.4095, 'brier': 0.5646, 'rps': 0.1972}
poisson DC  {'accuracy': 0.5384, 'macro_F1': 0.4026, 'brier': 0.5776, 'rps': 0.2031}
frequencies {'accuracy': 0.4442, 'macro_F1': 0.2051, 'brier': 0.645, 'rps': 0.2357}


In [13]:
from src.models import GRIDS

def rolling_eval(name, cols, params):
    Ps, ys = [], []
    for vs in seasons:
        tr = f[(f.season >= "1011") & (f.season < vs)]
        te = f[f.season == vs]
        _, P = fit_predict(name, tr, te, cols, **params)
        Ps.append(P); ys.append(te.label.values)
    return evaluate(np.concatenate(ys), np.vstack(Ps))

best = []
for s, cols in SETUPS.items():
    for name in ["LR", "RF", "XGB"]:
        trials = [{"params": p, **rolling_eval(name, cols, p)} for p in GRIDS[name]]
        top = min(trials, key=lambda r: r["rps"])
        best.append({"model": name, "setup": s, **top})
        print(name, s, top["params"], top["rps"])

tuned = pd.DataFrame(best)
tuned.to_csv("results/tuning_results.csv", index=False)
print(tuned.pivot(index="model", columns="setup", values="rps").round(4))

LR A {'C': 0.01} 0.21
RF A {'max_depth': 4, 'min_samples_leaf': 50} 0.2107
XGB A {'max_depth': 2, 'learning_rate': 0.03, 'n_estimators': 200, 'subsample': 0.8} 0.2117
LR B {'C': 0.01} 0.2027
RF B {'max_depth': None, 'min_samples_leaf': 20} 0.2042
XGB B {'max_depth': 2, 'learning_rate': 0.03, 'n_estimators': 200, 'subsample': 0.8} 0.2044
LR C {'C': 0.01} 0.2025
RF C {'max_depth': 6, 'min_samples_leaf': 20} 0.2043
XGB C {'max_depth': 2, 'learning_rate': 0.03, 'n_estimators': 200, 'subsample': 0.8} 0.2043
LR D {'C': 0.01} 0.2019
RF D {'max_depth': 6, 'min_samples_leaf': 20} 0.2027
XGB D {'max_depth': 2, 'learning_rate': 0.03, 'n_estimators': 200, 'subsample': 0.8} 0.2038
setup       A       B       C       D
model                                
LR     0.2100  0.2027  0.2025  0.2019
RF     0.2107  0.2042  0.2043  0.2027
XGB    0.2117  0.2044  0.2043  0.2038


In [14]:
extra = {"LR": [{"C": c} for c in (0.001, 0.003)],
         "XGB": [{"max_depth": d, "learning_rate": 0.01, "n_estimators": n, "subsample": 0.8}
                 for d in (1, 2) for n in (300, 600)]}
new = []
for r in tuned.to_dict("records"):
    for p in extra.get(r["model"], []):
        sc = rolling_eval(r["model"], SETUPS[r["setup"]], p)
        if sc["rps"] < r["rps"]:
            r = {**r, "params": p, **sc}
    new.append(r)
tuned = pd.DataFrame(new)
tuned.to_csv("results/tuning_results.csv", index=False)
print(tuned[["model", "setup", "params", "rps"]])

   model setup                                             params     rps
0     LR     A                                        {'C': 0.01}  0.2100
1     RF     A           {'max_depth': 4, 'min_samples_leaf': 50}  0.2107
2    XGB     A  {'max_depth': 1, 'learning_rate': 0.01, 'n_est...  0.2112
3     LR     B                                        {'C': 0.01}  0.2027
4     RF     B        {'max_depth': None, 'min_samples_leaf': 20}  0.2042
5    XGB     B  {'max_depth': 1, 'learning_rate': 0.01, 'n_est...  0.2037
6     LR     C                                        {'C': 0.01}  0.2025
7     RF     C           {'max_depth': 6, 'min_samples_leaf': 20}  0.2043
8    XGB     C  {'max_depth': 1, 'learning_rate': 0.01, 'n_est...  0.2037
9     LR     D                                       {'C': 0.003}  0.2015
10    RF     D           {'max_depth': 6, 'min_samples_leaf': 20}  0.2027
11   XGB     D  {'max_depth': 1, 'learning_rate': 0.01, 'n_est...  0.2031
